# Stage 2 Notebook 41 - Exp2LL Anchor + LineIoU regression + QFL

**The cls-collapse fix.** NB40 (Exp2KK) confirmed the anchor head holds the project all-time best geometry: matched_iou=0.483, oracle_f1=0.418. But pred_lanes still 1536 (= 192 priors x 8 batch) and decoded_f1=0.043 -- the binary cls task is degenerate. ASL gamma_neg=4 (NB40) and plain focal (NB39) both failed to separate the 5 matched priors from the 187 unmatched ones because the matching outcome is itself unstable across batches: the same prior is positive in some batches and negative in others depending on which competing priors win the dynamic-k IoU contest.

Exp2LL replaces binary {0, 1} matched_existence with continuous LineIoU regression: target = max LineIoU between this prior's CURRENT predicted curve (detached) and any valid GT lane in the same image. The cls head's output IS the ranking score at inference -- no more matching-instability confound.

`cls_loss_type: qfl` (Quality Focal Loss) weights BCE by `|target - sigmoid(logit)|^gamma` so the 95 percent of priors with target ~ 0 get near-zero gradient and cannot collapse all logits to 0 the way Exp2K (plain BCE) did.

All other settings = NB40 (anchor head, dynamic-k matching top-k=4, mask aux, cosine LR, uncertainty weighting, AMP, 20 epochs).

Reference: Li et al. 'Generalized Focal Loss V2' (2021). The published-recipe combination of continuous IoU target + QFL is what gives RTMDet/GFL their dense-prediction headroom.

### Run mode

1. Keep `DEBUG_MODE = True` for the first run.
2. After smoke + debug pass, change to `False` for the 20-epoch short run.
3. With AMP enabled, expect ~30 minutes wall-clock for 20 epochs at 3000 samples (matches NB40).
4. Output mirrored to notebook cell, Colab runtime log, Drive log file.
5. Do not rerun NB00.

In [1]:
import os, sys, subprocess, textwrap
from google.colab import drive
os.environ['PYTHONUNBUFFERED'] = '1'
drive.mount('/content/drive')

REPO_ROOT = '/content/drive/MyDrive/EcoCAR/yolop_vehicle_lane'
if not os.path.isdir(REPO_ROOT):
    raise FileNotFoundError(f'Missing project root: {REPO_ROOT}')
os.chdir(REPO_ROOT)
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'pyyaml', 'scipy', 'opencv-python-headless', 'tqdm', 'matplotlib'])
print('repo:', REPO_ROOT)

from stage2.scripts.notebook_utils import run_streaming
LOG_DIR = '/content/drive/MyDrive/EcoCAR/training_runs/notebook_logs'
os.makedirs(LOG_DIR, exist_ok=True)

Mounted at /content/drive
repo: /content/drive/MyDrive/EcoCAR/yolop_vehicle_lane


In [2]:
from pathlib import Path
import os, sys

CONFIG = 'stage2/configs/exp36_rmt_gca_anchor_iou_regression_joint.yaml'
LOG_FILE = os.path.join(LOG_DIR, f'{Path(CONFIG).stem}_smoke.log')
run_streaming([sys.executable, '-u', 'stage2/scripts/smoke_test_joint_models.py', CONFIG], log_path=LOG_FILE)

[run_streaming] command: /usr/bin/python3 -u stage2/scripts/smoke_test_joint_models.py stage2/configs/exp36_rmt_gca_anchor_iou_regression_joint.yaml
[run_streaming] log file: /content/drive/MyDrive/EcoCAR/training_runs/notebook_logs/exp36_rmt_gca_anchor_iou_regression_joint_smoke.log
OK exp36_rmt_gca_anchor_iou_regression_joint.yaml
  lane_shape=(1, 16, 72, 2) det_shape=(1, 4, 4)
  lane_loss=5.8965 det_loss=3.4200 grad_cos=0.1877 lambda_lane=0.0500
  gate_stats={'gate/det_mean': 0.4994601905345917, 'gate/lane_mean': 0.5017412304878235, 'gate/det_sat_low': 0.0, 'gate/det_sat_high': 0.0, 'gate/lane_sat_low': 0.0, 'gate/lane_sat_high': 0.0}
[run_streaming] return_code=0


0

In [3]:
from pathlib import Path
import os, sys

CONFIG = 'stage2/configs/exp36_rmt_gca_anchor_iou_regression_joint.yaml'
CURVE_TAR = '/content/drive/MyDrive/EcoCAR/datasets/bdd100k_clrkd_curve.tar'
CURVE_ROOT = '/content/bdd100k_clrkd_curve'

DEBUG_MODE = False

if DEBUG_MODE:
    RUN_TAG = 'debug'
    EPOCHS = 2
    BATCH_SIZE = 4
    LIMIT_TRAIN = 512
    LIMIT_VAL = 256
    PRINT_EVERY = 5
else:
    RUN_TAG = 'short20'
    EPOCHS = 20
    BATCH_SIZE = 8
    LIMIT_TRAIN = 3000
    LIMIT_VAL = 1000
    PRINT_EVERY = 5

run_stem = Path(CONFIG).stem + '_' + RUN_TAG
WORK_DIR = f'/content/{run_stem}'
OUTPUT_TAR = f'/content/drive/MyDrive/EcoCAR/training_runs/{run_stem}.tar'
LOG_FILE = os.path.join(LOG_DIR, f'{run_stem}_train.log')

cmd = [
    sys.executable, '-u', 'stage2/scripts/train_joint_model_experiment.py',
    '--config', CONFIG,
    '--curve-tar', CURVE_TAR,
    '--curve-root', CURVE_ROOT,
    '--work-dir', WORK_DIR,
    '--output-tar', OUTPUT_TAR,
    '--epochs', str(EPOCHS),
    '--batch-size', str(BATCH_SIZE),
    '--limit-train', str(LIMIT_TRAIN),
    '--limit-val', str(LIMIT_VAL),
    '--force-extract',
    '--print-every', str(PRINT_EVERY),
]

print('DEBUG_MODE:', DEBUG_MODE, flush=True)
print('About to run:', ' '.join(cmd), flush=True)
print('Output tar:', OUTPUT_TAR, flush=True)
print('Visible log file:', LOG_FILE, flush=True)
run_streaming(cmd, log_path=LOG_FILE)

DEBUG_MODE: False
About to run: /usr/bin/python3 -u stage2/scripts/train_joint_model_experiment.py --config stage2/configs/exp36_rmt_gca_anchor_iou_regression_joint.yaml --curve-tar /content/drive/MyDrive/EcoCAR/datasets/bdd100k_clrkd_curve.tar --curve-root /content/bdd100k_clrkd_curve --work-dir /content/exp36_rmt_gca_anchor_iou_regression_joint_short20 --output-tar /content/drive/MyDrive/EcoCAR/training_runs/exp36_rmt_gca_anchor_iou_regression_joint_short20.tar --epochs 20 --batch-size 8 --limit-train 3000 --limit-val 1000 --force-extract --print-every 5
Output tar: /content/drive/MyDrive/EcoCAR/training_runs/exp36_rmt_gca_anchor_iou_regression_joint_short20.tar
Visible log file: /content/drive/MyDrive/EcoCAR/training_runs/notebook_logs/exp36_rmt_gca_anchor_iou_regression_joint_short20_train.log
[run_streaming] command: /usr/bin/python3 -u stage2/scripts/train_joint_model_experiment.py --config stage2/configs/exp36_rmt_gca_anchor_iou_regression_joint.yaml --curve-tar /content/drive/M

0

## What to watch in Exp2LL training

Reference NB40 (anchor head, ASL cls, matched_existence): matched_iou=0.483, oracle_f1=0.418, decoded_f1=0.043, **pred_lanes=1536** (broken binary cls).

Pass criteria at epoch 20:
- **`pred_lanes < 400`** -- continuous QFL target should naturally produce a large gap between high-IoU and low-IoU priors so only ~50 priors per image cross the 0.3 decode threshold.
- **`val/lane/decoded_f1 >= 0.15`** -- 3x NB40, because the cls output now IS the ranking score we want.
- **`val/matched_line_iou >= 0.40`** -- preserves NB40's geometry champion.
- **`val/lane/decoded_oracle_f1 >= 0.35`** -- oracle ceiling stays high; the regression target should not distort geometry training.
- `train/grad_cosine_epoch_mean` logged each epoch (joint-conflict diagnostic).

Failure signals:
- decoded_f1 < 0.05 with low pred_lanes: QFL collapsed to all-zero (Exp2K failure mode). Drop `lineiou_target_pow` to 0.5.
- matched_iou drops below 0.30: cls regression is dragging geometry. Lower `w_cls` to 3.0.
- pred_lanes still > 1000: QFL didn't bite hard enough. Bump `qfl_gamma` to 3.0.
- decoded_f1 ~ oracle_f1: cls is now the bottleneck-free path; Exp2LL is the new champion.